# Edges, code-review-graph alone: guava (Java)
Baseline. graph.db opened read-only.

**Reference:** javac's parser (`java-ref/ListEdges.java`), parse only, not tree-sitter.

| Edge | What javac lists | Match key |
|---|---|---|
| CONTAINS | class / method and its parent | file + child start line + child name |
| INHERITS | `extends` / `implements` | file + class start line + parent simple name |
| IMPORTS_FROM | `import` lines | file + line |
| CALLS | `a.b()` and `new X()` call sites | file + line + called name |

Each edge is matched at most once. **CALLS is matched by name only**: which method is really called needs a full compile.

In [1]:
import pandas as pd, edge_lab as L
pd.set_option('display.max_colwidth', 90); pd.set_option('display.width', 250)
APP = 'guava'
REPO = L.ROOT / 'apps' / APP
DB = L.ROOT / 'results' / APP / 'runs/iter-003_2026-11-17_2247/crg/graph.db'
files = L.java_files(REPO)
R, errors = L.reference(REPO, files)
print('java files:', len(files), '| javac errors:', errors)
pd.Series({e: len(d) for e, d in R.items()}, name='expected (javac)')

java files: 3267 | javac errors: 0


CONTAINS         71080
INHERITS          6231
IMPORTS_FROM     39233
CALLS           301434
Name: expected (javac), dtype: int64

## 1. code-review-graph edges

In [2]:
G = L.crg_edges(DB, REPO)
pd.Series({e: len(d) for e, d in G.items()}, name='CRG edges (.java files)')

CONTAINS         66448
INHERITS          6001
IMPORTS_FROM     39233
CALLS           287769
Name: CRG edges (.java files), dtype: int64

## 2. Score

In [3]:
S, res = L.score_all(R, G, L.CRG_KEYS)
S

,expected (javac),reported,matched,missed,extra (false positive),found %,correct %
CONTAINS,71080.0,66448.0,66448.0,4632.0,0.0,93.5,100.0
INHERITS,6231.0,6001.0,6001.0,230.0,0.0,96.3,100.0
IMPORTS_FROM,39233.0,39233.0,39233.0,0.0,0.0,100.0,100.0
CALLS,301434.0,287769.0,286727.0,14707.0,1042.0,95.1,99.6


## 3. What failed: missed, by cause

In [4]:
m = res['CONTAINS'][0]; m = m[~m.hit]
print('CONTAINS missed'); display(m.assign(why=m.child.eq('').map({True: 'anonymous class'}).fillna(
    m.parent.eq('').map({True: 'method of anonymous class'})).fillna(m.kind)).why.value_counts())

CONTAINS missed


why
anonymous class              4444
method of anonymous class     107
method                         48
class                          33
Name: count, dtype: int64

In [5]:
m = res['INHERITS'][0]; m = m[~m.hit]
print('INHERITS missed:', len(m)); display(m[['file','line','cls','sup','how']].head(10))
g = G['INHERITS']; print('CRG INHERITS targets with a newline inside the name:', int(g.tgt.str.contains('\n').sum()))

INHERITS missed: 230


,file,line,cls,sup,how
52,android/guava-testlib/src/com/google/common/collect/testing/OneSizeTestContainerGenera...,35,OneSizeTestContainerGenerator,TestSubjectGenerator,extends
53,android/guava-testlib/src/com/google/common/collect/testing/OneSizeTestContainerGenera...,35,OneSizeTestContainerGenerator,TestContainerGenerator,extends
80,android/guava-testlib/src/com/google/common/collect/testing/TestCollectionGenerator.java,29,TestCollectionGenerator,TestContainerGenerator,extends
86,android/guava-testlib/src/com/google/common/collect/testing/TestListGenerator.java,29,TestListGenerator,TestCollectionGenerator,extends
88,android/guava-testlib/src/com/google/common/collect/testing/TestMapGenerator.java,29,TestMapGenerator,TestContainerGenerator,extends
89,android/guava-testlib/src/com/google/common/collect/testing/TestQueueGenerator.java,29,TestQueueGenerator,TestCollectionGenerator,extends
90,android/guava-testlib/src/com/google/common/collect/testing/TestSetGenerator.java,29,TestSetGenerator,TestCollectionGenerator,extends
91,android/guava-testlib/src/com/google/common/collect/testing/TestSortedMapGenerator.java,30,TestSortedMapGenerator,TestMapGenerator,extends
92,android/guava-testlib/src/com/google/common/collect/testing/TestSortedSetGenerator.java,29,TestSortedSetGenerator,TestSetGenerator,extends
273,android/guava-testlib/src/com/google/common/collect/testing/google/TestBiMapGenerator....,31,TestBiMapGenerator,TestContainerGenerator,extends


CRG INHERITS targets with a newline inside the name: 26


In [6]:
m = res['CALLS'][0]; m = m[~m.hit]
print('CALLS missed'); display(m.assign(why=m.name.isin(['super','this']).map({True: 'super(...) / this(...)'}).fillna('')
    .where(lambda s: s != '', m.how.map({'new': 'new X()', 'call': 'a.b() / b()'}))).why.value_counts())
m.head(10).assign(source=[L.source_line(REPO, f, l) for f, l in zip(m.head(10).file, m.head(10).line)])

CALLS missed


why
new X()                   6749
a.b() / b()               6681
super(...) / this(...)    1277
Name: count, dtype: int64

,file,line,start,name,how,in_method,hit,source
79,android/guava-testlib/src/com/google/common/collect/testing/AbstractIteratorTester.java,97,97,super,call,True,False,super(message);
85,android/guava-testlib/src/com/google/common/collect/testing/AbstractIteratorTester.java,118,118,super,call,True,False,"super(""Returned value '"" + actual + ""' not found. Remaining elements: "" + expected);"
117,android/guava-testlib/src/com/google/common/collect/testing/AbstractIteratorTester.java,276,276,KnownOrder,new,False,False,"KNOWN_ORDER,"
118,android/guava-testlib/src/com/google/common/collect/testing/AbstractIteratorTester.java,277,277,KnownOrder,new,False,False,UNKNOWN_ORDER
166,android/guava-testlib/src/com/google/common/collect/testing/AbstractIteratorTester.java,529,529,hasNext,call,True,False,"assertEquals(reference.hasNext(), target.hasNext());"
175,android/guava-testlib/src/com/google/common/collect/testing/AbstractIteratorTester.java,555,555,hasPrevious,call,True,False,"assertEquals(reference.hasPrevious(), target.hasPrevious());"
179,android/guava-testlib/src/com/google/common/collect/testing/AbstractIteratorTester.java,562,562,nextIndex,call,True,False,"assertEquals(reference.nextIndex(), target.nextIndex());"
183,android/guava-testlib/src/com/google/common/collect/testing/AbstractIteratorTester.java,569,569,previousIndex,call,True,False,"assertEquals(reference.previousIndex(), target.previousIndex());"
256,android/guava-testlib/src/com/google/common/collect/testing/AbstractMapTester.java,191,191,getKey,call,True,False,"""Wrong value for key "" + entry.getKey(), entry.getValue(), getMap().get(entry.getKey()));"
266,android/guava-testlib/src/com/google/common/collect/testing/AbstractMapTester.java,203,203,getKey,call,True,False,"if (Objects.equals(i.next().getKey(), newEntry.getKey())) {"


## 4. False positives (CRG edges with no javac match)

In [7]:
for e in L.EDGES:
    x = res[e][1]; x = x[~x.hit]
    print(e, 'false positives:', len(x))
x = res['CALLS'][1]; x = x[~x.hit].head(10)
x[['file','line','src','tgt']].assign(source=[L.source_line(REPO, f, l) for f, l in zip(x.file, x.line)])

CONTAINS false positives: 0
INHERITS false positives: 0
IMPORTS_FROM false positives: 0
CALLS false positives: 1042


,file,line,src,tgt,source
922,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java,289,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java::Map...,clear,"assertThrows(UnsupportedOperationException.class, map::clear);"
977,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java,742,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java::Map...,clear,"assertThrows(UnsupportedOperationException.class, entrySet::clear);"
1029,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java,480,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java::Map...,remove,"assertThrows(IllegalStateException.class, iterator::remove);"
1030,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java,483,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java::Map...,remove,"assertThrows(UnsupportedOperationException.class, iterator::remove);"
1254,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java,1249,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java::Map...,clear,"assertThrows(UnsupportedOperationException.class, keySet::clear);"
1471,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java,1514,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java::Map...,clear,"assertThrows(UnsupportedOperationException.class, valueCollection::clear);"
1484,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java,1342,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java::Map...,remove,"assertThrows(IllegalStateException.class, iterator::remove);"
1485,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java,1345,android/guava-testlib/src/com/google/common/collect/testing/MapInterfaceTest.java::Map...,remove,"assertThrows(UnsupportedOperationException.class, iterator::remove);"
2288,android/guava-testlib/src/com/google/common/collect/testing/SpliteratorTester.java,312,android/guava-testlib/src/com/google/common/collect/testing/SpliteratorTester.java::Sp...,add,"strategy.forEach(spliteratorSupplier.get(), resultsForStrategy::add);"
2309,android/guava-testlib/src/com/google/common/collect/testing/SpliteratorTester.java,288,android/guava-testlib/src/com/google/common/collect/testing/SpliteratorTester.java::Sp...,accept,"() -> new GeneralSpliteratorOfPrimitive<>(spliteratorSupplier.get(), c -> c::accept)));"
